# agent101 — Build Mini Claude Code from Zero

A step-by-step tutorial that teaches you how to build a mini AI coding agent from scratch.

Based on [learn-claude-code](https://github.com/shareAI-lab/learn-claude-code) by shareAI-lab.

---

# S01: The Agent Loop

`[ s01 ] s02 > s03 > s04 > s05 > s06 | s07 > s08 > s09 > s10 > s11 > s12`

> *"One loop & PowerShell is all you need"* — one tool + one loop = an agent.
>
> **Harness layer**: The loop — the model’s first connection to the real world.

## Problem

A language model can reason about code, but it can’t *touch* the real world — can’t read files, run tests, or check errors. Without a loop, every tool call requires you to manually copy-paste results back. **You become the loop.**

## Solution

```
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> |  Tool   |
| prompt |      |       |      | execute |
+--------+      +---+---+      +----+----+
                    ^                |
                    |   tool_result  |
                    +----------------+
                    (loop until model stops calling tools)
```

One exit condition controls the entire flow. The loop runs until the model stops calling tools.

### Step 1: Send a message to the LLM with a tool

First, let’s set up the client, define a `powershell` tool, and send a simple message to see how the LLM responds when it has a tool available.

In [1]:
import os
import json
import subprocess
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)
MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT")

SYSTEM = f"You are a coding agent at {os.getcwd()}. Use powershell to solve tasks. Act, don't explain."

TOOLS = [{
    "type": "function",
    "function": {
        "name": "powershell",
        "description": "Run a PowerShell command.",
        "parameters": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
        },
    },
}]

USER_MSG = "show current directory"
# Send a test message and inspect the raw response
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": USER_MSG},
    ],
    tools=TOOLS,
)

msg = response.choices[0].message
print("=== Raw Response ===")
print(f"content:    {msg.content}")
print(f"tool_calls: {msg.tool_calls}")
print()
if msg.tool_calls:
    tc = msg.tool_calls[0]
    print("=== First Tool Call ===")
    print(f"id:        {tc.id}")
    print(f"function:  {tc.function.name}")
    print(f"arguments: {tc.function.arguments}")

=== Raw Response ===
content:    None
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_axVVZDg9hyaKaMMb0SsgUMVe', function=Function(arguments='{"command":"Get-Location"}', name='powershell'), type='function')]

=== First Tool Call ===
id:        call_axVVZDg9hyaKaMMb0SsgUMVe
function:  powershell
arguments: {"command":"Get-Location"}


**What happened?** The LLM didn’t reply with text — it replied with a **tool call**. It wants us to run a PowerShell command.

But nobody executed it! The model can’t touch the real world on its own. That’s what the agent loop is for — we need code that:
1. Executes the tool call
2. Sends the result back to the LLM
3. Repeats until the model stops asking for tools

### Step 2: The Tool Handler

A simple function that executes a PowerShell command and returns the output. Includes basic safety checks and a timeout.



In [2]:
def run_powershell(command: str) -> str:
    """Execute a PowerShell command with safety guards."""
    dangerous = ["Remove-Item -Recurse -Force /", "shutdown", "Restart-Computer", "Stop-Computer"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(
            ["powershell", "-NoProfile", "-Command", command],
            cwd=os.getcwd(), capture_output=True, text=True, timeout=120,
        )
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"
    except (FileNotFoundError, OSError) as e:
        return f"Error: {e}"

# Quick test
print(run_powershell("Write-Output 'Hello from the agent!'"))

Hello from the agent!


### make real tool call
1. **User prompt** becomes the first message
2. **Send messages + tool definitions** to the LLM
3. **Check the response** — if the model didn’t call a tool, we’re done
4. **Execute each tool call**, collect results, append as messages, loop back to step 2

### Step 3: The Agent Loop


Let’s build it piece by piece.

This is the **entire secret** of an AI coding agent — a `while True` loop that:
1. Calls the LLM with the conversation history + tools
2. Appends the assistant’s response
3. If the model didn’t call any tools → **done** (exit)
4. Otherwise, execute each tool call, append results, and **loop back**

In [3]:
TOOL_HANDLERS = {"powershell": run_powershell}

def agent_loop(messages: list):
    """Core agent loop: call LLM, execute tools, repeat until done."""
    turn = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        # Show what we send to the LLM
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        # 1. Call the LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        # Show what the LLM returned
        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments})")
        else:
            print(f"  tool_calls: None (done!)")

        # 2. Append assistant turn
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        # 3. If no tool calls, we're done
        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        # 4. Execute each tool call, append results
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            cmd = args.get('command', '')
            print(f"\n  [exec] {name}: {cmd}")
            output = TOOL_HANDLERS[name](**args)
            print(f"  [result] {output[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": output,
            })

### What just happened?

That’s the **entire agent** in under 30 lines of core logic. Let’s trace the flow:

| Step | What happens |
|------|-------------|
| User types a prompt | Added to `messages` as `{"role": "user", ...}` |
| LLM receives messages + tools | Returns either text or tool calls |
| `msg.tool_calls` is not empty | We execute each tool, append results with `role: "tool"` |
| `msg.tool_calls` is empty/None | Model is done — we exit the loop |

The `messages` list grows with each iteration, giving the LLM full context of what it has done so far.

> **Key insight**: Everything else in this course layers on top of this loop — without changing it.

## Try It!

Run the cell below to start an interactive session. Try these prompts:
1. `Create a file called hello.py that prints "Hello, World!"`
2. `List all files in the current directory`
3. `What is the current git branch?`
4. `Create a directory called test_output and write 3 files in it`

Type `q` or `exit` to quit the REPL.

In [ ]:
# Interactive REPL — type your prompts, press Enter. Type "q" to quit.
history = []
while True:
    try:
        query = input("s01 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

s01 >>  create helloWorld.txt and add "Hello world" into it



Turn 1 - LLM Input (2 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101. Use powershell to solve...
  [1] user: create helloWorld.txt and add "Hello world" into it

Turn 1 - LLM Output:
  content:    None
  tool_call:  powershell({"command":"Set-Content -Path 'C:\\Users\\concao\\code\\agent101\\helloWorld.txt' -Value 'Hello world'"})

  [exec] powershell: Set-Content -Path 'C:\Users\concao\code\agent101\helloWorld.txt' -Value 'Hello world'
  [result] (no output)

Turn 2 - LLM Input (4 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101. Use powershell to solve...
  [1] user: create helloWorld.txt and add "Hello world" into it
  [2] assistant: (called 1 tool(s))
  [3] tool: (no output)

Turn 2 - LLM Output:
  content:    The file **helloWorld.txt** has been created and contains the text "Hello world".
  tool_calls: None (done!)

>>> Agent finished in 2 turn(s)



## What Changed

| Component     | Before     | After                          |
|---------------|------------|--------------------------------|
| Agent loop    | (none)     | `while True` + tool_calls check |
| Tools         | (none)     | `powershell` (one tool)        |
| Messages      | (none)     | Accumulating list              |
| Control flow  | (none)     | Exit when `tool_calls` is empty |

---

**Next: Session 02 — Tool Use** → expanding from one tool to a multi-tool dispatch system with file operations.